# cv-data-lesson ライブラリ受け入れテスト

授業用Notebookとは別の確認用Notebookです。GitHubの固定コミットからライブラリをインストールし、データ取得・ラベル・split・bbox・CPU/GPU境界をまとめて確認します。データ取得セルは固定Releaseから約650MBを取得します。

In [ ]:
LIBRARY_SOURCE_REF = "afbecc718f0bcb511d1f68b55fb8ddb774b8757a"
%pip install -q "cv-data-lesson[dataframe,display] @ git+https://github.com/708san/CV_data.git@afbecc718f0bcb511d1f68b55fb8ddb774b8757a"

In [ ]:
import importlib.metadata as metadata
from pathlib import Path
import cvlesson
from cvlesson import datasets

installed_version = metadata.version("cv-data-lesson")
print("version:", installed_version)
print("module:", cvlesson.__file__)
assert cvlesson.__file__ is not None
print("library source ref:", LIBRARY_SOURCE_REF)

In [ ]:
catalog = datasets.list_available()
catalog_ids = {item["id"] for item in catalog}
for required_id in ("geoset_ennea", "rdd2020_japan"):
    assert required_id in catalog_ids, required_id
print("catalog entries:", len(catalog), "required datasets: OK")

In [ ]:
# データをダウンロードせず、登録済みの元ラベルと日本語ラベルを確認する。
label_expectations = {
    "orange_fruit": {
        "cankered_orange": "傷んだオレンジ",
        "fresh_orange": "新鮮なオレンジ",
        "rotten_orange": "腐ったオレンジ",
    },
    "fruitseg30": {
        "Apple_Gala": "りんご（ガラ種）",
        "Apple_Golden Delicious": "りんご（ゴールデンデリシャス種）",
        "Green Coconut": "緑色のココナッツ",
        "Malta": "マルタオレンジ",
    },
    "indoor_waste_image": {"shell": "ナッツの殻"},
    "fire_recognition_image": {
        "Artificial_Fire": "擬似的な炎",
        "Fire": "炎",
        "Safe_Fire": "日常で扱われる火",
    },
    "fresh_rotten_fruits": {
        "fresh_apple": "新鮮なリンゴ",
        "rotten_apple": "傷んだリンゴ",
    },
    "agrifreshnet": {
        "Fresh Orange(1-9)": "新鮮なオレンジ",
        "Rotten Orange(20-35)": "傷んだオレンジ",
        "Semi fresh Orange(9-20)": "少し傷んだオレンジ",
    },
}
for dataset_id, expected in label_expectations.items():
    source = datasets._source(dataset_id)
    labels_ja = dict(zip(source.classes, source.class_names_ja))
    labels_ja.update(source.label_names_ja or {})
    for raw_label, expected_ja in expected.items():
        assert raw_label in source.classes, (dataset_id, raw_label)
        assert labels_ja[raw_label] == expected_ja, (dataset_id, raw_label, labels_ja[raw_label])
print("selected raw/Japanese labels: OK")

In [ ]:
CACHE_DIR = Path("/content/cvlesson-acceptance-cache")
geo = datasets.get("geoset_ennea", cache_dir=CACHE_DIR)
rdd = datasets.get("rdd2020_japan", cache_dir=CACHE_DIR)
print("GeoSetEnnea:", geo.splits)
print("RDD2020 Japan:", rdd.splits)
assert geo.splits == {"train": 84, "val": 22, "test": 30}
assert rdd.splits == {"train": 6299, "val": 802, "test": 799}
assert geo.root.is_dir() and rdd.root.is_dir()

In [ ]:
# 仕様: DatasetBundle.splitsとto_df()["split"]はvalへ統一し、source_splitに元表記を保持する。
# 物体検出のclass_name/class_name_jaはトップレベル列ではなく、各行のboxes内に保持する。
geo_df = geo.to_df()
rdd_df = rdd.to_df()
geo_splits = set(geo_df["split"].unique())
rdd_splits = set(rdd_df["split"].unique())
geo_source_splits = set(geo_df["source_split"].unique())
rdd_source_splits = set(rdd_df["source_split"].unique())
print("GeoSetEnnea normalized/source splits:", geo_splits, geo_source_splits)
print("RDD2020 normalized/source splits:", rdd_splits, rdd_source_splits)
assert geo_splits == {"train", "val", "test"}
assert rdd_splits == {"train", "val", "test"}
assert geo_source_splits == {"train", "val", "test"}
assert rdd_source_splits == {"train", "valid", "test"}
assert {"image_path", "label_file", "split", "boxes"} <= set(rdd_df.columns)
all_boxes = [box for boxes in rdd_df["boxes"] for box in boxes]
assert all(box.get("class_name") and box.get("class_name_ja") for box in all_boxes)
print("DataFrame schema and original split notation: OK")

In [ ]:
# RDD2020の全bboxが有限値で、画像境界内（極小の丸め誤差を除く）にあることを確認する。
bbox_count = 0
for boxes in rdd_df["boxes"]:
    for box in boxes:
        bbox_count += 1
        values = [float(box[key]) for key in ("x_center", "y_center", "width", "height")]
        assert all(value == value and abs(value) != float("inf") for value in values)
        x_center, y_center, width, height = values
        assert -1e-9 <= x_center - width / 2 <= 1 + 1e-9
        assert -1e-9 <= y_center - height / 2 <= 1 + 1e-9
        assert -1e-9 <= x_center + width / 2 <= 1 + 1e-9
        assert -1e-9 <= y_center + height / 2 <= 1 + 1e-9
assert bbox_count == 16469
print("RDD2020 bbox count/range: OK (", bbox_count, ")")

In [ ]:
from cvlesson import generation, models

model_devices = models.available_devices()
print("ResNet devices:", model_devices)
assert model_devices["cpu"] is True
assert models.ResNetClassifier(device="cuda", weights=None).device == "cuda"
generation_devices = generation.available_devices()
print("Generation devices:", generation_devices)
assert generation_devices["cpu"] is True  # CPUはTiny-SDの実行先
try:
    generation.load(model="sdxl", device="cpu")
except generation.GenerationError as error:
    print("明示SDXLのCPU拒否: OK -", error)
else:
    raise AssertionError("明示したSDXLがCPUで実行可能と誤って判定されました")

In [ ]:
# PaddleOCR本体をダウンロードせず、WebPをPNGへデコードしてから渡す経路を確認する。
try:
    from PIL import Image
    import tempfile
    from cvlesson.ocr import OCR
except ImportError as error:
    print("OCR WebP smoke skipped:", error)
else:
    class ProbePipeline:
        def __init__(self):
            self.paths = []
        def predict(self, *, input):
            self.paths.append(input)
            return [{"res": {"rec_texts": ["検証"], "rec_scores": [0.99], "rec_polys": [[[1, 1], [20, 1], [20, 10], [1, 10]]]}}]
    with tempfile.TemporaryDirectory() as temporary_directory:
        webp_path = Path(temporary_directory) / "sample.webp"
        Image.new("RGB", (32, 32), "white").save(webp_path, format="WEBP")
        probe = ProbePipeline()
        result = OCR(probe, device="cpu", language="japan").predict(webp_path)
        assert result.text == "検証"
        assert probe.paths and probe.paths[0].endswith("input.png")
    print("OCR WebP decode path: OK")

In [ ]:
# 共通表示APIの実動作を確認する。
import tempfile
from pathlib import Path
from IPython.display import display
import numpy as np
from PIL import Image
from cvlesson import display as cv_display
from cvlesson.ocr import OCRLine, OCRResult
from cvlesson.vision import Detection, Segmentation
display_fixture = Image.new("RGB", (128, 128), "white")
display_detection = Detection("fixture", 0.9, (16, 20, 96, 110))
display_mask = np.zeros((128, 128), dtype=bool)
display_mask[32:96, 32:96] = True
display_segmentation = Segmentation(display_mask, 0.8, "box", display_detection.box)
display_ocr = OCRResult((OCRLine("確認", 0.95, ((8, 8), (70, 8), (70, 28), (8, 28))),), "fixture")
rendered = cv_display.render(display_fixture, [display_detection, display_segmentation, display_ocr], max_detections=5, max_ocr_lines=5)
assert rendered.size == display_fixture.size
with tempfile.TemporaryDirectory() as temporary_directory:
    saved_path = cv_display.save(display_fixture, Path(temporary_directory) / "overlay.png", [display_detection, display_segmentation, display_ocr], max_detections=5, max_ocr_lines=5)
    assert saved_path.is_file()
shown = cv_display.show(display_fixture, [display_detection, display_segmentation, display_ocr], max_detections=5, max_ocr_lines=5)
assert shown.size == display_fixture.size
print("display render/show/save: OK")

## 判定

ここまで全セルが成功すれば、指定コミットのライブラリをColabへ導入し、固定ReleaseのGeoSetEnnea/RDD2020、ラベル表記、元split保持、bbox検証、`auto`の利用可能性、明示SDXLのCPU拒否、WebP入力経路を確認できています。ResNetの実学習やDETR/SAM/Tiny-SD/SDXLの実モデル確認は、`auto_device_model_acceptance_test.ipynb`で実行します。